In [ ]:
# Setup ngrok tunnel for API access
print("🌐 Setting up ngrok tunnel...")

# Install pyngrok
!pip install -q pyngrok

# Set ngrok auth token (optional but recommended)
# Get your token from https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = ""  # ← Add your ngrok auth token here

from pyngrok import ngrok, conf

if NGROK_AUTH_TOKEN:
    conf.get_default().auth_token = NGROK_AUTH_TOKEN
    print("✅ Using authenticated ngrok tunnel")
else:
    print("⚠️  Using anonymous tunnel (limited to 1 connection)")
    print("   Get free auth token: https://dashboard.ngrok.com/get-started/your-authtoken")

# Kill any existing tunnels
ngrok.kill()

# Start API server in background
import subprocess
import time

print("\n🚀 Starting FastAPI server...")
api_process = subprocess.Popen(
    ['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='/content/egyption_id_ready',
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for server to start
time.sleep(5)

# Create tunnel
public_url = ngrok.connect(8000)
print(f"\n✅ API server started!")
print(f"🔗 Public API URL: {public_url}")
print(f"📖 API Docs: {public_url}/docs")
print(f"📖 API Health: {public_url}/")

# Keep this cell running to maintain the tunnel
# To stop: press the stop button or run: ngrok.kill()

## 📦 Part 1: Environment Setup

### 1.1 Check GPU and Mount Google Drive

In [ ]:
# Check GPU
!nvidia-smi

# Check Python version
!python --version

# Mount Google Drive (optional - for saving outputs)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

print("✅ Environment check complete!")

### 1.2 Clone Repository and Install Dependencies

In [ ]:
# Clone the repository
%cd /content
!git clone https://github.com/your-repo/egyption_id_ready.git
%cd /content/egyption_id_ready

# Install system dependencies
!apt-get update && apt-get install -y libgl1-mesa-glx libglib2.0-0 libgomp1 curl wget git

print("✅ Repository cloned and system dependencies installed!")

In [ ]:
# Clone PaddleOCR repository (required for training)
%cd /content
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
print("✅ PaddleOCR cloned!")

In [ ]:
# Install Python dependencies
!pip install -q -r requirements.txt
!pip install -q google-generativeai

print("✅ Python dependencies installed!")

In [ ]:
# Helper function for robust downloads
def download_with_retry(url, output_path, max_retries=3, timeout=60):
    """Download file with retry logic."""
    import subprocess
    from pathlib import Path
    
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    
    for attempt in range(max_retries):
        try:
            result = subprocess.run(
                ['wget', '-q', '--timeout=' + str(timeout), 
                 '--tries=1', '-O', str(output), url],
                capture_output=True,
                text=True,
                timeout=timeout * 2
            )
            if result.returncode == 0 and output.exists() and output.stat().st_size > 0:
                return True
        except Exception as e:
            print(f"Attempt {attempt + 1}/{max_retries} failed: {e}")
    
    return False

print("✅ Download helper loaded!")

### 1.3 Verify Installation

In [ ]:
import torch, cv2, pandas as pd, numpy as np

print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"OpenCV: {cv2.__version__}")
print(f"Pandas: {pd.__version__}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("\n✅ Installation verified!")

## 📥 Part 2: Download Models

### 2.1 Download YOLO Detection Models from GitHub

In [ ]:
!mkdir -p /content/egyption_id_ready/weights

print("⬇️ Downloading Card Detection Model...")
!wget -q --show-progress -O /content/egyption_id_ready/weights/card_detection.pt https://github.com/NASO7Y/OCR_Egyptian_ID/raw/main/detect_id_card.pt

print("\n⬇️ Downloading Field Detection Model...")
!wget -q --show-progress -O /content/egyption_id_ready/weights/field_detection.pt https://github.com/NASO7Y/OCR_Egyptian_ID/raw/main/detect_odjects.pt

print("\n⬇️ Downloading NID Detection Model...")
!wget -q --show-progress -O /content/egyption_id_ready/weights/nid_detection.pt https://github.com/NASO7Y/OCR_Egyptian_ID/raw/main/detect_id.pt

print("\n📦 Model Files:")
!ls -lh /content/egyption_id_ready/weights/

In [ ]:
# Download Arabic dictionary file (required for OCR)
!wget -q -O /content/egyption_id_ready/arabic_dict.txt \
    https://raw.githubusercontent.com/NAMO7Y/Egyptian_ID_OCR/main/arabic_dict.txt

# Verify file exists
from pathlib import Path
dict_path = Path('/content/egyption_id_ready/arabic_dict.txt')
if dict_path.exists():
    print(f"✅ Arabic dictionary downloaded: {dict_path.stat().st_size} bytes")
else:
    print("❌ Arabic dictionary download failed!")
    # Create minimal dictionary as fallback
    with open(dict_path, 'w', encoding='utf-8') as f:
        # Write basic Arabic characters
        arabic_chars = ''.join([chr(i) for i in range(0x0600, 0x06FF)])
        f.write(arabic_chars)
    print("⚠️  Created minimal fallback dictionary")

In [ ]:
# Download field detector ONNX model (required for inference)
!mkdir -p /content/egyption_id_ready/model

# Try to download from project repository
# Note: Replace with actual model URL if different
!wget -q --timeout=30 --tries=3 -O /content/egyption_id_ready/model/field_detector.onnx \
    https://github.com/NASO7Y/OCR_Egyptian_ID/raw/main/field_detector.onnx || \
    echo "⚠️  Field detector model not available - will use YOLO models instead"

# Verify download
from pathlib import Path
model_path = Path('/content/egyption_id_ready/model/field_detector.onnx')
if model_path.exists() and model_path.stat().st_size > 0:
    print(f"✅ Field detector model downloaded: {model_path.stat().st_size / 1024 / 1024:.2f} MB")
else:
    print("⚠️  Field detector model not available - two-stage detection will use YOLO models only")

In [ ]:
# Validate all required models are downloaded
from pathlib import Path

ROOT = Path('/content/egyption_id_ready')
required_files = {
    'weights/card_detection.pt': 'Card Detection Model',
    'weights/field_detection.pt': 'Field Detection Model',
    'weights/nid_detection.pt': 'NID Detection Model',
    'arabic_PP-OCRv3_rec/best_accuracy.pdparams': 'PaddleOCR Model',
    'arabic_dict.txt': 'Arabic Dictionary',
}

print("🔍 Validating downloaded files...")
all_ok = True
for file_path, description in required_files.items():
    full_path = ROOT / file_path
    if full_path.exists() and full_path.stat().st_size > 0:
        size_mb = full_path.stat().st_size / 1024 / 1024
        print(f"  ✅ {description}: {size_mb:.2f} MB")
    else:
        print(f"  ❌ {description}: MISSING")
        all_ok = False

if all_ok:
    print("\n✅ All models downloaded successfully!")
else:
    print("\n⚠️  Some models are missing. Check error messages above.")

In [ ]:
# Monitor disk space
!df -h /content

# Clean unnecessary files to save space
print("\n🧹 Cleaning unnecessary files...")
!rm -rf /content/egyption_id_ready/.git 2>/dev/null || true
!rm -rf /content/PaddleOCR/.git 2>/dev/null || true
!rm -rf /content/egyption_id_ready/__pycache__ 2>/dev/null || true
!rm -rf /content/egyption_id_ready/src/__pycache__ 2>/dev/null || true
print("✅ Cleanup complete!")

# Show freed space
!df -h /content

### 2.2 Download OCR Models

In [ ]:
print("⬇️ Preparing EasyOCR models (Arabic + English)...")
!mkdir -p /content/egyption_id_ready/models_cache

import easyocr
reader = easyocr.Reader(['ar', 'en'], gpu=True, model_storage_directory='/content/egyption_id_ready/models_cache')
print("✅ EasyOCR initialized!")

print("\n⬇️ Downloading PaddleOCR pretrained model...")
!mkdir -p /content/egyption_id_ready/arabic_PP-OCRv3_rec
!wget -q --show-progress -O /content/egyption_id_ready/arabic_PP-OCRv3_rec/best_accuracy.pdparams https://paddleocr.bj.bcebos.com/PP-OCRv3/arabic/rec_arabic_ppocr_v3_train/best_accuracy.pdparams

print("\n✅ OCR models ready!")

## 📊 Part 3: Download Dataset

### 3.1 Download Egyptian ID Dataset from Google Drive

**Dataset Link**: Replace with your Google Drive link

The dataset should contain:
- `train/images/` - Training images + labels
- `valid/images/` - Validation images + labels  
- `test/images/` - Test images + labels

In [ ]:
FILE_ID = "YOUR_DATASET_FILE_ID_HERE"
OUTPUT_PATH = "/content/egyption_id_ready/dataset.zip"

print(f"⬇️ Downloading dataset from Google Drive...")
!pip install -q gdown
!gdown --id {FILE_ID} -O {OUTPUT_PATH}

print("\n✅ Dataset downloaded!")

In [ ]:
print("📦 Extracting dataset...")
!unzip -q /content/egyption_id_ready/dataset.zip -d /content/egyption_id_ready/

print("\n📊 Dataset Statistics:")
for split in ['train', 'valid', 'test']:
    img_count = !ls /content/egyption_id_ready/{split}/images/*.jpg 2>/dev/null | wc -l
    lbl_count = !ls /content/egyption_id_ready/{split}/labels/*.txt 2>/dev/null | wc -l
    print(f"   {split:6s}: {img_count[0]:>6} images, {lbl_count[0]:>6} labels")

print("\n✅ Dataset ready!")

## 🔧 Part 4: Build Dataset with Two-Stage Detection

In [ ]:
import sys
from pathlib import Path
ROOT = Path('/content/egyption_id_ready')
sys.path.insert(0, str(ROOT))
print(f"📂 Project root: {ROOT}")

In [ ]:
import cv2, matplotlib.pyplot as plt
from src.card_detector import load_card_detector

print("🔧 Loading two-stage detector...")
detector = load_card_detector(
    card_model_path=str(ROOT / 'weights' / 'card_detection.pt'),
    field_model_path=str(ROOT / 'weights' / 'field_detection.pt'),
)
print("✅ Detector loaded!")

test_images = list((ROOT / 'train' / 'images').glob('*.jpg'))
if test_images:
    image = cv2.imread(str(test_images[0]))
    card_crop, fields = detector.detect_full(image, translate_to_project=True)
    print(f"✂️  Card crop: {card_crop.shape[1]}x{card_crop.shape[0]}")
    print(f"📋 Fields detected: {len(fields)}")

In [ ]:
print("🚀 Processing full dataset...")
!python /content/egyption_id_ready/scripts/process_full_dataset_two_stage.py --splits train valid test
print("\n✅ Dataset processing complete!")
!ls -lh /content/egyption_id_ready/rec/images/two_stage/ | head -5

In [ ]:
# Validate dataset processing results
from pathlib import Path
import pandas as pd

print("🔍 Validating dataset processing...")

# Check cropped images
crops_dir = Path('/content/egyption_id_ready/rec/images/two_stage')
if crops_dir.exists():
    crop_count = len(list(crops_dir.glob('*.jpg')))
    print(f"  ✅ Cropped fields: {crop_count:,} images")
else:
    print("  ❌ Crops directory not found!")

# Check metadata CSV
metadata_path = Path('/content/egyption_id_ready/crops_metadata_two_stage.csv')
if metadata_path.exists():
    df = pd.read_csv(metadata_path)
    print(f"  ✅ Metadata: {len(df):,} records")
    print(f"     Fields: {df['field'].nunique()} unique")
    print(f"     Avg confidence: {df['confidence'].mean():.3f}")
else:
    print("  ❌ Metadata CSV not found!")

print("\n✅ Dataset validation complete!")

## 🏷️ Part 5: Label Crops with OCR

In [ ]:
OCR_METHOD = "qari-airllm"
print(f"🏷️ Labeling crops with {OCR_METHOD}...")

!python /content/egyption_id_ready/scripts/label_crops.py --method {OCR_METHOD} --splits train valid

print("\n✅ Labeling complete!")
!wc -l /content/egyption_id_ready/crops_labeled.csv

## 🎯 Part 6: Prepare Training Data

In [ ]:
print("📝 Preparing training data...")
!python /content/egyption_id_ready/scripts/prepare_paddle_labels.py --input /content/egyption_id_ready/crops_labeled.csv --output-dir /content/egyption_id_ready/rec
print("\n✅ Training data prepared!")
!head -3 /content/egyption_id_ready/rec/train.txt

## 🚀 Part 7: Train PaddleOCR Model

In [ ]:
# Check for existing checkpoints before training
from pathlib import Path
checkpoint_dir = Path('/content/egyption_id_ready/output/egyptian_id_rec')
latest_checkpoint = checkpoint_dir / 'latest.pdparams'

if latest_checkpoint.exists():
    print(f"📋 Found existing checkpoint: {latest_checkpoint}")
    print("⏸️  Training will resume from last checkpoint")
else:
    print("🆕 No checkpoint found - starting fresh training")

print("\n🚀 Starting PaddleOCR training (2-4 hours)...")
print("📊 Checkpoints saved every 5 epochs")
print("⏱️  Use Colab's runtime disconnect prevention if available")

# Run training
!cd /content/egyption_id_ready && python PaddleOCR/tools/train.py -c configs/egyptian_id_rec.yml -o Global.use_gpu=true -o Global.epoch_num=100 -o Global.save_model_dir=/content/egyption_id_ready/output/egyptian_id_rec/

print("\n✅ Training complete!")

## 📊 Part 8: Evaluate Model

In [ ]:
print("📊 Evaluating model...")
!python /content/egyption_id_ready/scripts/evaluate.py --model-path /content/egyption_id_ready/output/egyptian_id_rec/best_accuracy --test-data /content/egyption_id_ready/rec/test.txt
print("\n✅ Evaluation complete!")

## 📦 Part 9: Export to ONNX

In [ ]:
!mkdir -p /content/egyption_id_ready/onnx
!python /content/egyption_id_ready/PaddleOCR/tools/export_model.py -c /content/egyption_id_ready/configs/egyptian_id_rec.yml -o Global.pretrained_model=/content/egyption_id_ready/output/egyptian_id_rec/best_accuracy
!paddle2onnx --model_dir /content/egyption_id_ready/inference/rec --save_file /content/egyption_id_ready/onnx/rec.onnx --opset_version 11
!python -m onnxsim /content/egyption_id_ready/onnx/rec.onnx /content/egyption_id_ready/onnx/rec_sim.onnx
print("\n✅ ONNX export complete!")
!ls -lh /content/egyption_id_ready/onnx/

## 🔌 Part 10: Test Inference

In [ ]:
sys.path.insert(0, '/content/egyption_id_ready')
from src.inference import EgyptianIDOCR

ocr = EgyptianIDOCR(
    det_onnx=str(ROOT / 'model' / 'field_detector.onnx'),
    rec_onnx=str(ROOT / 'onnx' / 'rec_sim.onnx'),
    dict_path=str(ROOT / 'arabic_dict.txt'),
    use_gpu=True,
)
print("✅ OCR loaded!")

test_images = list((ROOT / 'test' / 'images').glob('*.jpg'))
if test_images:
    image = cv2.imread(str(test_images[0]))
    results = ocr.extract(image)
    for field_name, field_data in results.items():
        print(f"{field_name}: {field_data.get('text', 'N/A')}")

## 💾 Part 11: Save to Google Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/egyption_id_ready/output
!cp -r /content/egyption_id_ready/output/egyptian_id_rec/ /content/drive/MyDrive/egyption_id_ready/output/
!cp -r /content/egyption_id_ready/onnx/ /content/drive/MyDrive/egyption_id_ready/
!cp /content/egyption_id_ready/crops_labeled.csv /content/drive/MyDrive/egyption_id_ready/
print("\n✅ All outputs saved to Google Drive!")

## 📊 Summary

In [ ]:
print("=" * 60)
print("🎉 EGYPTIAN ID OCR - COLAB NOTEBOOK COMPLETE!")
print("=" * 60)
print("\n✅ Accomplished:")
print("   1. Environment setup with GPU")
print("   2. Downloaded YOLO + OCR models")
print("   3. Processed dataset")
print("   4. Labeled crops with OCR")
print("   5. Trained PaddleOCR")
print("   6. Evaluated model")
print("   7. Exported to ONNX")
print("   8. Tested inference")
print("   9. Saved to Google Drive")
print("\n📂 Outputs:")
        print(f"   - Model: /content/egyption_id_ready/output/")
        print(f"   - ONNX: /content/egyption_id_ready/onnx/")
        print(f"   - Drive: /content/drive/MyDrive/egyption_id_ready/")
        print("=" * 60)

In [ ]:
# Final validation and summary
from pathlib import Path
import pandas as pd

print("=" * 60)
print("🎉 FINAL VALIDATION")
print("=" * 60)

ROOT = Path('/content/egyption_id_ready')

# Check all outputs
outputs = {
    'Trained Model': ROOT / 'output' / 'egyptian_id_rec' / 'best_accuracy.pdparams',
    'ONNX Model': ROOT / 'onnx' / 'rec_sim.onnx',
    'Labeled Data': ROOT / 'crops_labeled.csv',
    'Metadata': ROOT / 'crops_metadata_two_stage.csv',
}

print("\n📂 Output Files:")
all_ok = True
for name, path in outputs.items():
    if path.exists():
        size_mb = path.stat().st_size / 1024 / 1024
        print(f"  ✅ {name}: {size_mb:.2f} MB")
    else:
        print(f"  ❌ {name}: MISSING")
        all_ok = False

# Check Google Drive backup
drive_path = Path('/content/drive/MyDrive/egyption_id_ready')
if drive_path.exists():
    print(f"\n💾 Google Drive Backup:")
    for item in drive_path.rglob('*'):
        if item.is_file():
            print(f"  ✅ {item.relative_to(drive_path)}")

print("\n" + "=" * 60)
if all_ok:
    print("✅ ALL VALIDATIONS PASSED!")
    print("\n🚀 Your Egyptian ID OCR system is ready!")
    print("\n📋 Next Steps:")
    print("   1. Download models from Google Drive")
    print("   2. Deploy API using Docker or locally")
    print("   3. Integrate with your application")
else:
    print("⚠️  Some outputs are missing. Check error messages above.")
print("=" * 60)